# SAM Audio Separation Pipeline
Automated Soundtrack Personalisation Project

In [ ]:
!pip install git+https://github.com/facebookresearch/sam-audio.git -q
!pip install torchaudio librosa soundfile matplotlib yt-dlp -q
print('Setup complete')

In [ ]:
import os
import torch
import torchaudio
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from sam_audio import SAMAudio, SAMAudioProcessor

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
YOUTUBE_URL = 'https://www.youtube.com/watch?v=YOUR_VIDEO_ID'
INPUT_PATH = '/content/input.wav'

!yt-dlp -x --audio-format wav -o "{INPUT_PATH}" "{YOUTUBE_URL}"

display(Audio(INPUT_PATH))

In [ ]:
MODEL_ID = 'facebook/sam-audio-small'

model = SAMAudio.from_pretrained(MODEL_ID).to(device).eval()
processor = SAMAudioProcessor.from_pretrained(MODEL_ID)

print('Model loaded')

In [ ]:
SEPARATIONS = {
    'dialogue': 'a person speaking',
    'music': 'background music',
    'sfx': 'sound effects'
}

OUTPUT_DIR = '/content/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

results = {}

for name, prompt in SEPARATIONS.items():
    inputs = processor(audios=[INPUT_PATH], descriptions=[prompt]).to(device)

    with torch.no_grad():
        result = model.separate(inputs)

    target_path = f'{OUTPUT_DIR}/{name}.wav'

    torchaudio.save(target_path, result.target[0].unsqueeze(0).cpu(), processor.audio_sampling_rate)

    results[name] = target_path

    print(f'Saved {name}')

In [ ]:
print('Original')
display(Audio(INPUT_PATH))

for name, path in results.items():
    print(name)
    display(Audio(path))

In [ ]:
dialogue, sr = librosa.load(results['dialogue'], sr=None)
background, _ = librosa.load(INPUT_PATH, sr=None)

min_len = min(len(dialogue), len(background))

mix = dialogue[:min_len]*1.5 + background[:min_len]*0.3
mix = mix / max(abs(mix))

OUT = '/content/personalised.wav'
sf.write(OUT, mix, sr)

display(Audio(OUT))

In [ ]:
signals = [
    (INPUT_PATH, 'Original'),
    (results['dialogue'], 'Dialogue'),
    (results['music'], 'Music'),
    (OUT, 'Personalised')
]

fig, axs = plt.subplots(len(signals), 1, figsize=(10, 12))

for ax, (path, title) in zip(axs, signals):
    y, sr = librosa.load(path, sr=None)
    D = librosa.amplitude_to_db(abs(librosa.stft(y)), ref=max)
    librosa.display.specshow(D, sr=sr, ax=ax)
    ax.set_title(title)

plt.tight_layout()
plt.show()